# SpectralViT on ADNI3: three tasks, three bases

This notebook trains three spectral vision transformers from scratch on 100 ADNI3 subjects each, one per basis family, and animates them on held-out subjects:

| Task | Basis | Tokens |
|---|---|---|
| Sex classification | Principal components (empirical) | whitened PCA coefficients, count by inner cross-validation |
| Age estimation | Block cosine (harmonic) | 12 mm blocks, one token per frequency holding every block's coefficient |
| Cognitive status, CN vs MCI | Surface graph Laplacian (graphical) | age, sex and three SynthSeg volumes, plus 8 surface modes fit to the remaining error |

Every model sees the whole brain: each T1 is affinely registered to one reference subject and resampled to a 96³ grid at 2 mm (a 192 mm box).

**Inputs.** ADNI3 in BIDS layout (`sub-XXX/anat/sub-XXX_T1w.nii.gz` and `participants.tsv`) and SynthSeg 2.0 outputs for every subject (see the SynthSeg section). A CUDA GPU.

**Paths.** Set `ADNI_ROOT` and `ADNI_WORK` in the environment before starting Jupyter, or edit the first code cell.

**Outputs.** Everything is written under `WORK`: registered volumes and transforms, the SynthSeg volume table, model outputs, and the animated figure. Each expensive step reuses its saved output when one exists. Set `SMOKE=1` in the environment for a quick end-to-end check with 5 training epochs.

Raw ADNI images and per-subject derivatives must stay off public hosts under the ADNI Data Use Agreement; figures of representative cases are fine.

In [ ]:
import os, sys, time
ADNI_ROOT = os.environ.get('ADNI_ROOT', '/path/to/ADNI3')                 # BIDS root, read only
WORK = os.environ.get('ADNI_WORK', '/path/to/derivatives/ADNI')           # derivatives: registered volumes, SynthSeg outputs, figure
SYNTHSEG = os.path.join(WORK, 'synthseg', 'seg')        # <id>_synthseg.nii.gz and <id>_vol.csv per subject
FIG = os.path.join(WORK, 'demo'); os.makedirs(FIG, exist_ok=True)
SMOKE = os.environ.get('SMOKE') == '1'
N_TRAIN = 100                                           # training subjects per task
EPOCHS = 5 if SMOKE else 500                            # paper recipe: 500 epochs, one-cycle schedule
N_FRAMES = 3 if SMOKE else 24                           # held-out subjects shown in the animation

In [ ]:
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F, scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.ndimage import gaussian_filter, map_coordinates, binary_dilation, center_of_mass
from scipy.spatial import cKDTree
from scipy.fft import dctn, idctn
from skimage.filters import threshold_otsu, threshold_multiotsu
from skimage.measure import marching_cubes
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt, imageio.v2 as imageio
from networks import SpectralViT
dev = torch.device('cuda')

## Cohort

Labels come from `participants.tsv`. Cognitive status is CN against MCI (early, late and unspecified MCI pooled); SMC and AD are left out of that task. Fifty CN or MCI subjects are held out for every task, and one of them is the registration reference.

In [ ]:
def t1_path(sid): return f'{ADNI_ROOT}/{sid}/anat/{sid}_T1w.nii.gz'
P = pd.read_csv(f'{ADNI_ROOT}/participants.tsv', sep='\t')
P = P[[os.path.exists(t1_path(s)) for s in P.ID]].reset_index(drop=True)
IDS, SEX, AGE, GRP = P.ID.values, P.Sex.values, P.Age.values.astype(np.float32), P.Group.values
N = len(IDS)
ysex = (SEX == 'F').astype(int)
ycog = np.where(GRP == 'CN', 0, np.where(np.isin(GRP, ['EMCI', 'LMCI', 'MCI']), 1, -1))

rng = np.random.RandomState(1)
cogpool = np.where(ycog >= 0)[0]
held = np.sort(rng.choice(cogpool, 50, replace=False))
REF = int(held[7])                                      # registration reference, not scored
rest = held[held != REF]                                # 49 held-out subjects
train_all, train_cog = np.setdiff1d(np.arange(N), held), np.setdiff1d(cogpool, held)
print(f'{N} subjects; groups {dict(zip(*np.unique(GRP, return_counts=True)))}; reference {IDS[REF]}; {len(rest)} held out')

## Registration

Each T1 is registered to the reference with a 12-parameter affine on the GPU, maximizing normalized cross-correlation. A rigid plus isotropic-scale stage aligns the whole head on blurred, downsampled images, then a penalized affine stage refines it on the central 96 mm. The start is the head's center of mass. When that start ends below an NCC of 0.45, eight more starts (head height ±15 to 45 mm, front-back ±20 mm) are tried and the best kept; this catches scans whose head sits low in the field of view.

The model input is a 96³ volume at 2 mm spacing covering a 192 mm box around the brain. Its center is offset by (−2, −1, −3) mm from the reference's field-of-view center, so the display plane 17 mm left of center is exactly voxel plane 40.

In [ ]:
V, WB_SP, WB_CEN = 96, 2.0, np.array([-2.0, -1.0, -3.0])
WB_FILE, WB_META = f'{WORK}/adni_vols_96_wb2mm.npy', f'{WORK}/adni_vols_96_wb2mm_meta.npz'

def load_t1(i):
    im = nib.load(t1_path(IDS[i])); v = np.asarray(im.dataobj, dtype=np.float32)
    return np.clip(v / (np.percentile(v, 99.5) + 1e-8), 0, 1.2), np.array(im.header.get_zooms()[:3], dtype=np.float32)

def grid(shape, center, extent):
    # voxel offsets of a box of `extent` voxels centered at `center`, as (..., 3)
    ax = [torch.arange(e, device=dev, dtype=torch.float32) - (e - 1) / 2 + c for e, c in zip(extent, center)]
    return torch.stack(torch.meshgrid(*ax, indexing='ij'), -1)

def sample(mov, pts):
    # trilinear sample of mov (1, 1, X, Y, Z) at voxel coordinates pts (..., 3)
    sz = torch.tensor(mov.shape[2:], device=dev, dtype=torch.float32)
    return F.grid_sample(mov, (pts / ((sz - 1) / 2) - 1).flip(-1)[None], mode='bilinear', padding_mode='zeros', align_corners=True)[0, 0]

def ncc(a, b):
    a = a - a.mean(); b = b - b.mean(); return (a * b).sum() / (a.norm() * b.norm() + 1e-8)

def com(v): return np.array(center_of_mass(v > threshold_otsu(v)))

def register(mov_np, fix_np, c_mov, t0, ratio, big_np):
    """Affine from reference offsets (mm) to moving voxels: p = (x @ A.T + t) * ratio + c_mov. Returns A, t and the NCC on the 1 mm crop."""
    ratio = torch.tensor(ratio, device=dev, dtype=torch.float32)
    mov = torch.tensor(mov_np, device=dev)[None, None]; fix = torch.tensor(fix_np, device=dev)
    movb = torch.tensor(gaussian_filter(mov_np, 1.5), device=dev)[None, None]; bigb = torch.tensor(gaussian_filter(big_np, 1.5), device=dev)
    w = torch.zeros(3, device=dev, requires_grad=True); ls = torch.zeros(1, device=dev, requires_grad=True)      # rotation vector, log isotropic scale
    dA = torch.zeros(3, 3, device=dev, requires_grad=True); t = torch.tensor(t0, device=dev, dtype=torch.float32, requires_grad=True)
    def Amat(affine):
        Wx = torch.stack([torch.stack([0 * w[0], -w[2], w[1]]), torch.stack([w[2], 0 * w[0], -w[0]]), torch.stack([-w[1], w[0], 0 * w[0]])])
        R = torch.linalg.matrix_exp(Wx) * torch.exp(ls); return R + dA if affine else R
    # coarse levels align the whole head (160 mm box, blurred); fine levels the 96 mm crop; translation gets a 10x step
    for scale, iters, lr, affine, region in ((8, 200, 5e-2, False, 'big'), (4, 200, 2e-2, False, 'big'), (4, 100, 1e-2, True, 'big'), (2, 100, 5e-3, True, 'crop'), (1, 60, 2e-3, True, 'crop')):
        src, fsrc = (movb, bigb) if region == 'big' else (mov, fix)
        fx = fsrc[::scale, ::scale, ::scale]; ext = fx.shape; pts = grid(ext, torch.zeros(3, device=dev), ext) * scale
        opt = torch.optim.Adam([{'params': [w, ls] + ([dA] if affine else []), 'lr': lr}, {'params': [t], 'lr': 10 * lr}])
        for _ in range(iters):
            opt.zero_grad(); p = (pts @ Amat(affine).T + t) * ratio + c_mov
            loss = -ncc(sample(src, p), fx) + 1e-1 * (dA ** 2).sum() + 1e-1 * ls ** 2; loss.backward(); opt.step()
    with torch.no_grad():
        A = Amat(True); p = (grid((V, V, V), torch.zeros(3, device=dev), (V, V, V)) @ A.T + t) * ratio + c_mov
        return A.cpu().numpy(), t.detach().cpu().numpy(), float(ncc(sample(mov, p), fix))

if os.path.exists(WB_FILE) and os.path.exists(WB_META):
    X = np.load(WB_FILE, mmap_mode='r'); WM = dict(np.load(WB_META)); assert list(WM['ids']) == list(IDS) and int(WM['ref']) == REF
    print(f'loaded registered volumes: median NCC {np.median(WM["ncc"]):.2f}')
else:
    ref, z_ref = load_t1(REF); c_ref = np.array(ref.shape) // 2; r = V // 2; com_ref = com(ref) - c_ref
    RB = 80; big = np.zeros((2 * RB,) * 3, np.float32); lo, hi = c_ref - RB, c_ref + RB
    src_sl = tuple(slice(max(l, 0), min(h, n)) for l, h, n in zip(lo, hi, ref.shape)); dst_sl = tuple(slice(max(-l, 0), max(-l, 0) + (s.stop - s.start)) for l, s in zip(lo, src_sl)); big[dst_sl] = ref[src_sl]
    fix = ref[c_ref[0]-r:c_ref[0]+r, c_ref[1]-r:c_ref[1]+r, c_ref[2]-r:c_ref[2]+r].copy()
    pts_wb = grid((V,) * 3, torch.zeros(3, device=dev), (V,) * 3) * WB_SP + torch.tensor(WB_CEN, device=dev, dtype=torch.float32)
    STARTS = [(0, 0)] + [(0, dz) for dz in (-45, -30, -15, 15, 30, 45)] + [(-20, 0), (20, 0)]   # (front-back, head height) offsets in mm
    vols, As, ts, nccs = [], [], [], []; t0 = time.time()
    for i in range(N):
        mov, z_mov = load_t1(i); c_mov = torch.tensor(np.array(mov.shape) // 2, device=dev, dtype=torch.float32); ratio = z_ref / z_mov; cm = np.array(mov.shape) // 2
        base = (com(mov) - cm) / ratio - com_ref; best = None
        for dy, dz in STARTS:
            A, t, s = register(mov, fix, c_mov, base + np.array([0.0, dy, dz]), ratio, big)
            if best is None or s > best[2]: best = (A, t, s)
            if (dy, dz) == (0, 0) and s >= 0.45: break
        A, t, s = best
        with torch.no_grad():   # 2 mm sampling from ~1 mm data: blur first
            p = (pts_wb @ torch.tensor(A, device=dev).T + torch.tensor(t, device=dev)) * torch.tensor(ratio, device=dev) + c_mov
            v = sample(torch.tensor(gaussian_filter(mov, 0.85), device=dev)[None, None], p).cpu().numpy()
        vols.append(((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)); As.append(A); ts.append(t); nccs.append(s)
        if i % 25 == 0: print(f'  {i}/{N}  {time.time() - t0:.0f}s', flush=True)
    np.save(WB_FILE, np.stack(vols)); WM = dict(ids=IDS, ref=REF, ncc=np.array(nccs), A=np.stack(As), t=np.stack(ts), center_mm=WB_CEN, spacing_mm=WB_SP)
    np.savez(WB_META, **WM); X = np.load(WB_FILE, mmap_mode='r'); print(f'registered {N} subjects; median NCC {np.median(nccs):.2f}')
Z_REF = load_t1(REF)[1]

## SynthSeg volumes

SynthSeg 2.0 (FreeSurfer 7.4) runs once per subject on the CPU, outside this notebook; one SLURM array task per subject takes about 3 minutes with 4 threads and 24 GB of memory:

```
mri_synthseg --i <T1> --o <WORK>/synthseg/seg/<id>_synthseg.nii.gz --vol <WORK>/synthseg/seg/<id>_vol.csv --qc <WORK>/synthseg/seg/<id>_qc.csv --threads 4 --cpu
```

Left and right volumes are summed and divided by intracranial volume. Subjects without a complete output are left out of the cognitive status task.

In [ ]:
rows = []
for sid in IDS:
    f = f'{SYNTHSEG}/{sid}_vol.csv'
    if not (os.path.exists(f) and os.path.exists(f'{SYNTHSEG}/{sid}_synthseg.nii.gz')): rows.append(pd.Series(dtype=float, name=sid)); continue
    v = pd.read_csv(f); v.columns = [c.strip() for c in v.columns]; v = v.iloc[0, 1:].astype(float); r = {'tiv': v['total intracranial']}
    for c in v.index.drop('total intracranial'):
        k = c.replace('left ', '').replace('right ', ''); r[k] = r.get(k, 0.0) + v[c]
    rows.append(pd.Series(r, name=sid))
VT = pd.DataFrame(rows)
MTL = ['hippocampus', 'amygdala', 'inferior lateral ventricle']                        # medial temporal volumes used for cognitive status
SV = {k: (VT[k] / VT['tiv'] * 1000).values.astype(np.float32) for k in MTL}           # mL per liter of intracranial volume
print(f'SynthSeg volumes for {VT.tiv.notna().sum()} of {N} subjects')

## Training sets

One draw of 100 training subjects per task, stratified by label for the two classification tasks.

In [ ]:
def draw(n, pool, y, seed=0):
    r = np.random.RandomState(seed)
    if y is None: return np.sort(r.choice(pool, n, replace=False))
    keep = []
    for c in np.unique(y[pool]):
        p = pool[y[pool] == c]; keep += list(r.choice(p, size=int(round(n * len(p) / len(pool))), replace=False))
    keep = np.array(sorted(keep))
    if len(keep) < n: keep = np.sort(np.concatenate([keep, r.choice(np.setdiff1d(pool, keep), n - len(keep), replace=False)]))
    return keep[:n]
TR = {'sex': draw(N_TRAIN, train_all, ysex), 'age': draw(N_TRAIN, train_all, None), 'cog': draw(N_TRAIN, train_cog[np.isfinite(VT['tiv'].values[train_cog])], ycog)}
Xrest = np.asarray(X[rest], dtype=np.float32); Xref = np.asarray(X[REF], dtype=np.float32)

## Bases

**Principal components** are fit on each training set inside the sex model below.

**Block cosine.** The volume is cut into 16³ blocks of 6 voxels (12 mm). Each block keeps its lowest 4³ cosine frequencies, sign-log transformed. Token *t* holds frequency *t* (ranked by frequency magnitude) for every block, ordered from the center outward, so the model attends over frequencies.

**Surface graph Laplacian.** On the reference subject, the brain is masked with its SynthSeg labels (dilated 2 mm), and isosurfaces are taken at the three multi-Otsu thresholds inside it: the outer brain boundary, CSF to gray, and gray to white. The triangle meshes are joined into one graph by linking vertices on different surfaces that lie within 8 mm, and any remaining island is tied to the main component through each vertex's nearest neighbor there. The tokens are a subject's intensities at the vertices projected on the lowest eigenvectors of the normalized graph Laplacian.

In [ ]:
# block cosine layout, shared by the tokens and the display
GRID, BZ, FQ = 16, V // 16, 4; nb, nf = GRID ** 3, FQ ** 3
kz, ky, kx = np.unravel_index(np.arange(nf), (FQ,) * 3); forder = np.argsort(kz ** 2 + ky ** 2 + kx ** 2, kind='stable')
bc = np.array(np.unravel_index(np.arange(nb), (GRID,) * 3)).T + 0.5; border = np.argsort(((bc - GRID / 2) ** 2).sum(1), kind='stable')
PERM = np.array([b * nf + forder[r] for r in range(nf) for b in border]); INV = np.argsort(PERM)

def dct_tokens(vols):
    out = np.zeros((len(vols), nb * nf), np.float32)
    for i, v in enumerate(vols):
        b = np.asarray(v).reshape(GRID, BZ, GRID, BZ, GRID, BZ).transpose(0, 2, 4, 1, 3, 5)
        c = dctn(b, type=2, norm='ortho', axes=(3, 4, 5))[..., :FQ, :FQ, :FQ].reshape(nb * nf)
        out[i] = np.sign(c) * np.log1p(np.abs(c))
    return out[:, PERM]

In [ ]:
def mesh_graph(verts, faces):
    e = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]]); e = np.vstack([e, e[:, ::-1]])
    W = sp.csr_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(verts),) * 2); W.data[:] = 1; return W

def link(W, rows, cols):
    L_ = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=W.shape); return W.maximum(L_.maximum(L_.T))

# reference brain mask in the 2 mm box
t1_ref = nib.load(t1_path(IDS[REF])); sg_ref = nib.load(f'{SYNTHSEG}/{IDS[REF]}_synthseg.nii.gz'); c_ref = np.array(t1_ref.shape) // 2
off = (np.arange(V) - (V - 1) / 2) * WB_SP
g = np.array(np.meshgrid(*[c_ref[d] + off + WB_CEN[d] for d in range(3)], indexing='ij')).reshape(3, -1)
q = np.linalg.inv(sg_ref.affine) @ t1_ref.affine @ np.vstack([g, np.ones(g.shape[1])])
BM = binary_dilation(map_coordinates(np.asarray(sg_ref.dataobj).astype(np.int16), q[:3], order=0, mode='constant', cval=0).reshape(V, V, V) > 0, iterations=1)

# isosurfaces at the three multi-Otsu thresholds inside the mask
t1 = np.clip(Xref, np.percentile(Xref[BM], 1), np.percentile(Xref[BM], 99)); ths = threshold_multiotsu(t1[BM], classes=4)
smooth = gaussian_filter(np.where(BM, t1, t1[BM].min()), 1.0)
vs, fs, labs, offset = [], [], [], 0
for k, th in enumerate(ths):
    v, f, _, _ = marching_cubes(smooth, th, step_size=1)
    inner = ((v > 1.5) & (v < V - 2.5)).all(1); f = f[inner[f].all(1)]; keep = np.unique(f)
    remap = -np.ones(len(v), int); remap[keep] = np.arange(len(keep)); v, f = v[keep], remap[f]
    vs.append(v); fs.append(f + offset); labs.append(np.full(len(v), k)); offset += len(v)
verts, faces, slab = np.vstack(vs), np.vstack(fs), np.concatenate(labs); NV = len(verts)

# one graph: mesh edges, links between surfaces within 4 voxels (8 mm), every island tied to the main component
W = mesh_graph(verts, faces); rows, cols = [], []
for a in range(len(ths)):
    for b in range(len(ths)):
        if a == b: continue
        ia, ib = np.where(slab == a)[0], np.where(slab == b)[0]; d, j = cKDTree(verts[ib]).query(verts[ia]); ok = d < 4
        rows += list(ia[ok]); cols += list(ib[j[ok]])
W = link(W, rows, cols)
nc, comp = sp.csgraph.connected_components(W); main = np.bincount(comp).argmax()
for c in range(nc):
    if c == main: continue
    ic, im_ = np.where(comp == c)[0], np.where(comp == main)[0]; _, j = cKDTree(verts[im_]).query(verts[ic]); rows += list(ic); cols += list(im_[j])
W = link(W, rows, cols); deg = np.asarray(W.sum(1)).ravel(); Dm = sp.diags(1 / np.sqrt(deg))
K_SURF = 8
_, Us = eigsh(sp.eye(NV) - Dm @ W @ Dm + 1e-6 * sp.eye(NV), k=K_SURF + 1, sigma=0, which='LM'); Us = np.asarray(Us)[:, 1:]
def surf_tokens(vols): return np.stack([map_coordinates(np.asarray(v), verts.T, order=1, mode='nearest') @ Us for v in vols])
print(f'surface graph: {NV} vertices on {len(ths)} surfaces, {sp.csgraph.connected_components(W)[0]} component')

## Models

All three are `SpectralViT` models from `networks.py`, trained from scratch with AdamW and a one-cycle schedule. Capacity is picked by 3-fold cross-validation inside each training set. Attention is the last layer's, averaged over heads and queries, scaled by the rank weights for scalar tokens.

When scalar covariates are mixed with spectral modes, the covariates go first at rank weight 1 and the modes keep 1/1, 1/2, … after them. Appended after the modes, a covariate would start at roughly 1/k of the first mode's strength.

In [ ]:
T = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)

def make_model(n_in, embed, layers, patch=1, rank=True, mode=True, ncov=0):
    torch.manual_seed(0)
    m = SpectralViT(n_inputs=n_in, patch_size=patch, embed_dim=embed, n_layers=layers, n_heads=2, use_rank_weights=rank, use_mode_weights=mode, use_pos_embed=False).to(dev)
    if ncov and rank:
        with torch.no_grad(): m.rank_weights[:ncov] = 1.0; m.rank_weights[ncov:] = 1.0 / torch.arange(1, n_in - ncov + 1, device=dev, dtype=torch.float32)
    if not mode: m.use_pos_embed = True; m.pos_embed = nn.Parameter(torch.randn(n_in // patch, 1, embed, device=dev) * 0.02)
    return m

def train(m, Z, y, loss, lr, batch, wd=0.0, epochs=EPOCHS, offset=None):
    cols = [T(Z), T(y)] + ([T(offset)] if offset is not None else [])
    ld = DataLoader(TensorDataset(*cols), batch_size=batch, shuffle=True)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd); sch = OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(ld), epochs=epochs)
    for _ in range(epochs):
        m.train()
        for b in ld:
            opt.zero_grad(); out = m(b[0], return_logit=True) + (b[2] if offset is not None else 0); l = loss(out, b[1])
            if not torch.isfinite(l): continue
            l.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step()
    return m.eval()

def predict(m, Z):
    with torch.no_grad(): return m(T(Z), return_logit=True).cpu().numpy()

def attend(m, Z):
    last = m.transformer.layers[-1]; store = {}; orig = last.self_attn.forward
    def hooked(q, k, v, **kw):
        kw.update(need_weights=True, average_attn_weights=True); o, w = orig(q, k, v, **kw); store['w'] = w.detach(); return o, w
    last.self_attn.forward = hooked
    try: p = m(T(Z), return_logit=True).detach().cpu().numpy()   # gradients stay on so the fused fast path, which drops the weights, is skipped
    finally: last.self_attn.forward = orig
    A = store['w'].mean(1).cpu().numpy()
    if m.use_rank_weights and m.patch_size == 1: A = A * np.abs(m.rank_weights[:A.shape[1]].detach().cpu().numpy())[None]
    return A, p

def inner_cv(cands, Z, y, fit_eval, higher=True):
    scores = {}
    for c in cands:
        s = [fit_eval(c, tr, va) for tr, va in KFold(3, shuffle=True, random_state=0).split(Z)]; scores[c] = float(np.mean(s)); print(f'  capacity {c}: inner {scores[c]:.3f}', flush=True)
    return (max if higher else min)(scores, key=scores.get)

### Sex classification: principal components

Whitened PCA fit on the training set; the number of components is picked from 16, 32 and 64 by inner cross-validation. Four layers, embedding 16, rank and mode weights, binary cross-entropy, learning rate 1e-4, batch 8.

In [ ]:
kp = TR['sex']; Xs = np.asarray(X[kp], dtype=np.float32).reshape(len(kp), -1); Xe = Xrest.reshape(len(rest), -1); bce = nn.BCEWithLogitsLoss()
def pca_fit(c, A): mu = A.mean(0); return mu, PCA(n_components=c, whiten=True, random_state=0).fit(A - mu)
def sex_eval(c, tr, va):
    mu, p = pca_fit(c, Xs[tr]); m = train(make_model(c, 16, 4), p.transform(Xs[tr] - mu), ysex[kp][tr], bce, 1e-4, 8)
    return roc_auc_score(ysex[kp][va], predict(m, p.transform(Xs[va] - mu)))
k_sex = 16 if SMOKE else inner_cv([16, 32, 64], Xs, ysex[kp], sex_eval)
mu_sex, pca = pca_fit(k_sex, Xs); Zt_sex, Zr_sex = pca.transform(Xs - mu_sex), pca.transform(Xe - mu_sex)
a_sex, p_sex = attend(train(make_model(k_sex, 16, 4), Zt_sex, ysex[kp], bce, 1e-4, 8), Zr_sex)
print(f'sex: {k_sex} components, held-out AUC {roc_auc_score(ysex[rest], p_sex):.3f}')

### Age estimation: block cosine

Each of the 64 tokens is one frequency across all 4,096 blocks, standardized per feature. Two layers, a learned position embedding, embedding width 16 or 32 by inner cross-validation, L1 loss on the standardized age, learning rate 1e-3, weight decay 0.05, batch 4. Reported as mean absolute error in years.

In [ ]:
kp = TR['age']; Ft = dct_tokens(np.asarray(X[kp], dtype=np.float32)); Fe = dct_tokens(Xrest)
fm, fsd = Ft.mean(0), Ft.std(0) + 1e-6; am, asd = AGE[kp].mean(), AGE[kp].std() + 1e-6
Zt_age, Zr_age, yt_age = (Ft - fm) / fsd, (Fe - fm) / fsd, (AGE[kp] - am) / asd
l1 = nn.L1Loss()
def age_model(c): return make_model(nb * nf, c, 2, patch=nb, mode=False)
def age_eval(c, tr, va):
    m = train(age_model(c), Zt_age[tr], yt_age[tr], l1, 1e-3, 4, wd=0.05); return float(np.mean(np.abs(predict(m, Zt_age[va]) - yt_age[va]))) * asd
e_age = 16 if SMOKE else inner_cv([16, 32], Zt_age, yt_age, age_eval, higher=False)
a_age, p_age = attend(train(age_model(e_age), Zt_age, yt_age, l1, 1e-3, 4, wd=0.05), Zr_age); p_age = p_age * asd + am
print(f'age: embedding {e_age}, held-out MAE {np.mean(np.abs(p_age - AGE[rest])):.2f} y (mean estimate {np.mean(np.abs(AGE[rest] - am)):.2f} y)')

### Cognitive status: covariates, then surface graph Laplacian on the residual

Stage 1 is a transformer on five scalar tokens: age, sex, and hippocampus, amygdala and temporal horn volumes. Stage 2 is a transformer on the 8 surface modes, trained to fit stage 1's remaining error on the logit scale, with stage 1's prediction as a fixed offset. The training offsets are cross-fitted, so stage 2 learns from stage 1's out-of-fold errors. Stage 2 is regularized (100 epochs, weight decay 0.05) and enters scaled by 0.25, 0.5 or 1, picked by inner cross-validation.

Stage 2 is there to show where a surface model looks; cross-validation that allowed a weight of zero would choose it, because the surface modes add little beyond the volumes. Report stage 1 as the result.

In [ ]:
kp = TR['cog']; yc = ycog[kp].astype(np.float32)
cm, csd = AGE[kp].mean(), AGE[kp].std() + 1e-6; sm, ssd = ysex[kp].mean(), ysex[kp].std() + 1e-6; vm = {k: (SV[k][kp].mean(), SV[k][kp].std() + 1e-6) for k in MTL}
def cov(idx): return np.stack([(AGE[idx] - cm) / csd, (ysex[idx] - sm) / ssd] + [(SV[k][idx] - vm[k][0]) / vm[k][1] for k in MTL], 1)
NCOV = 2 + len(MTL); Ct, Ce = cov(kp), cov(rest)
Lt, Le = surf_tokens(np.asarray(X[kp], dtype=np.float32)), surf_tokens(Xrest); ml, msd = Lt.mean(0), Lt.std(0) + 1e-6; Mt, Me = (Lt - ml) / msd, (Le - ml) / msd
E2 = 5 if SMOKE else 100

def stage1(C, y): return train(make_model(NCOV, 16, 4, ncov=NCOV), C, y, bce, 1e-4, 8)
def stage2(M, y, off): return train(make_model(K_SURF, 16, 4), M, y, bce, 1e-4, 8, wd=0.05, epochs=E2, offset=off)
def crossfit(C, y):
    off = np.zeros(len(y), np.float32)
    for a, b in StratifiedKFold(3, shuffle=True, random_state=0).split(C, y): off[b] = predict(stage1(C[a], y[a]), C[b])
    return off

GAM = (0.25, 0.5, 1.0); gam = 0.5
if not SMOKE:
    cvs = {g: [] for g in GAM}
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=1).split(Ct, yc):
        off_va = predict(stage1(Ct[tr], yc[tr]), Ct[va]); r_va = predict(stage2(Mt[tr], yc[tr], crossfit(Ct[tr], yc[tr])), Mt[va])
        for g in GAM: cvs[g].append(roc_auc_score(yc[va], off_va + g * r_va))
    gam = max(GAM, key=lambda g: np.mean(cvs[g])); print('  inner AUC by residual weight:', {g: round(float(np.mean(cvs[g])), 3) for g in GAM})
a1, p_cog1 = attend(stage1(Ct, yc), Ce)
a2, r_cog = attend(stage2(Mt, yc, crossfit(Ct, yc)), Me); p_cog = p_cog1 + gam * r_cog
print(f'cognitive status: stage 1 held-out AUC {roc_auc_score(ycog[rest], p_cog1):.3f}; with the surface residual at weight {gam}: {roc_auc_score(ycog[rest], p_cog):.3f}')

## Results

In [ ]:
err = p_age - AGE[rest]
summary = pd.DataFrame({'held-out (49)': [f'AUC {roc_auc_score(ysex[rest], p_sex):.3f}', f'MAE {np.mean(np.abs(err)):.2f} y, RMSE {np.sqrt(np.mean(err ** 2)):.2f} y', f'AUC {roc_auc_score(ycog[rest], p_cog1):.3f}'],
                        'baseline': ['AUC 0.5', f'mean estimate: MAE {np.mean(np.abs(AGE[rest] - am)):.2f} y', 'AUC 0.5']},
                       index=['Sex classification', 'Age estimation', 'Cognitive status (stage 1)'])
summary

## Animated figure

Each frame is one held-out subject on a parasagittal plane 17 mm left of the reference midline. The slice, the subject's brain outline (SynthSeg labels without CSF) and its hippocampus, amygdala and temporal horn are carried onto the plane through that subject's own registration. Top row: the subject in each basis. Bottom row: each model's attention, projected back to the image through its basis; for cognitive status, the surface model's attention with the three volume tokens' attention painted on their structures.

In [ ]:
S, PSAG, PLANE = 256, -17.0, 40                         # display slice (1 mm), its offset from the reference center, and the matching voxel plane of the 2 mm box
DY, DZ = slice(31, 223), slice(29, 221)                 # where the 192 mm box lands on the display slice
def up(a): return np.kron(a, np.ones((2, 2)))
show = np.array([i for i in rest[:N_FRAMES + 1] if IDS[i] != 'sub-046'][:N_FRAMES])   # sub-046's scan stops inside the cerebellum

def display_data(i):
    """Slice, SynthSeg labels on the plane and the projected left temporal horn, through subject i's transform."""
    im = nib.load(t1_path(IDS[i])); v = np.asarray(im.dataobj, dtype=np.float32); v = np.clip(v / (np.percentile(v, 99.5) + 1e-8), 0, 1.2)
    ratio = Z_REF.astype(np.float64) / np.array(im.header.get_zooms()[:3]); c_mov = np.array(v.shape) // 2; A, t = WM['A'][i].astype(np.float64), WM['t'][i].astype(np.float64)
    o = np.arange(S) - (S - 1) / 2
    def plane(dx): yy, zz = np.meshgrid(o, o, indexing='ij'); return np.stack([np.full_like(yy, dx), yy, zz], -1).reshape(-1, 3)
    to_vox = lambda P_: (P_ @ A.T + t) * ratio + c_mov
    sg = nib.load(f'{SYNTHSEG}/{IDS[i]}_synthseg.nii.gz'); L = np.asarray(sg.dataobj).astype(np.int16); v2s = np.linalg.inv(sg.affine) @ im.affine
    def labels(P_): q = v2s @ np.vstack([to_vox(P_).T, np.ones(len(P_))]); return map_coordinates(L, q[:3], order=0, mode='constant', cval=0)
    sl = map_coordinates(v, to_vox(plane(PSAG)).T, order=1, mode='constant', cval=0).reshape(S, S); lab = labels(plane(PSAG)).reshape(S, S)
    horn = np.zeros(S * S, bool)
    for dx in range(-48, 0): horn |= np.isin(labels(plane(dx)), (5, 44))
    return np.clip(sl, 0, 1.2), lab, horn.reshape(S, S)

In [ ]:
# reconstructions and attention images on the display plane
pca_imgs = pca.components_.reshape(k_sex, V, V, V)[:, PLANE]
def pca_recon(z): return (mu_sex + pca.mean_ + (z * np.sqrt(pca.explained_variance_)) @ pca.components_).reshape(V, V, V)[PLANE]

def dct_coeffs(z): c = (z * fsd + fm)[INV]; return np.sign(c) * np.expm1(np.abs(c))
xb, xo = PLANE // BZ, PLANE % BZ                         # block row and in-block offset of the plane
def dct_recon(z):
    c = dct_coeffs(z).reshape(GRID, GRID, GRID, FQ, FQ, FQ)[xb]; U = np.zeros((GRID, GRID, BZ, BZ, BZ)); U[..., :FQ, :FQ, :FQ] = c
    blk = idctn(U, type=2, norm='ortho', axes=(2, 3, 4))[:, :, xo]                      # (GRID, GRID, BZ, BZ) in-plane block images
    return blk.transpose(0, 2, 1, 3).reshape(V, V)
def dct_attention(a, z):
    # token t holds frequency forder[t] in every block; its image is that frequency's contribution across the plane's blocks
    c = dct_coeffs(z).reshape(GRID, GRID, GRID, nf)[xb]; out = np.zeros((V, V))
    for tok in range(nf):
        f = forder[tok]; U = np.zeros((BZ, BZ, BZ)); U[np.unravel_index(f, (FQ,) * 3)] = 1; basis = idctn(U, type=2, norm='ortho')[xo]
        out += a[tok] * np.einsum('yz,ab->yazb', c[:, :, f], basis).reshape(V, V)
    out = np.abs(out); return (out / (np.percentile(out, 99.5) + 1e-8)) ** 0.7

# surface values painted at 1 mm: inverse-distance blend of the 6 nearest vertices within 3 mm, inside the subject's own brain
py, pz = np.meshgrid(np.arange(DY.start, DY.stop), np.arange(DZ.start, DZ.stop), indexing='ij')
P3 = np.column_stack([np.full(py.size, PLANE, float), (py.ravel() - DY.start - 0.5) / 2, (pz.ravel() - DZ.start - 0.5) / 2])
dpx, jpx = cKDTree(verts).query(P3, k=6); near = dpx[:, 0] < 1.5; wpx = 1.0 / (dpx + 0.25); wpx /= wpx.sum(1, keepdims=True)
def paint(vals, brain): m = near & brain; img = np.full(py.size, np.nan); img[m] = (vals[jpx[m]] * wpx[m]).sum(1); return img.reshape(py.shape)
Pm = sp.diags(1 / deg) @ W
def smooth_surf(v, it=3):
    for _ in range(it): v = Pm @ v
    return v
SHR = a1[:, 2:] / a1.sum(1, keepdims=True); SMIN, SMAX = np.percentile(SHR, [1, 99])          # volume-token attention share, one scale for every frame

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9}); TXT, BG, EDGE = '#E6EDF3', '#0E1116', '#242C36'
def frame_axes(ax, title):
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(DY.start - 0.5, DY.stop - 0.5); ax.set_ylim(DZ.start - 0.5, DZ.stop - 0.5)
    ax.set_title(title, color=TXT, fontsize=9); [s.set_edgecolor(EDGE) for s in ax.spines.values()]
def gray(ax, img): ax.imshow(np.clip(img, 0, 1).T, cmap='gray', origin='lower', vmin=0, vmax=1)
def in_box(rec): out = np.zeros((S, S)); r = rec - np.percentile(rec, 0.5); out[DY, DZ] = up(r / (np.percentile(r, 99.5) + 1e-8)); return out
def overlay(ax, img_box, cmap='inferno', alpha=0.6, lo=0, hi=1, upsample=True):
    o = np.full((S, S), np.nan); o[DY, DZ] = up(img_box) if upsample else img_box; ax.imshow(o.T, cmap=cmap, origin='lower', alpha=alpha, vmin=lo, vmax=hi)
norm01 = lambda im: im / (np.percentile(im, 99.5) + 1e-8)

frames = []
for j, i in enumerate(show):
    ii = int(np.where(rest == i)[0][0]); sl, lab, horn = display_data(i); brain = ((lab > 0) & (lab != 24))[DY, DZ].ravel()
    fig = plt.figure(figsize=(9.6, 6.9), dpi=100); fig.patch.set_facecolor(BG)
    gs = fig.add_gridspec(2, 3, wspace=0.08, hspace=0.14, left=0.03, right=0.97, top=0.88, bottom=0.08)
    ax = fig.add_subplot(gs[0, 0]); gray(ax, in_box(pca_recon(Zr_sex[ii]))); frame_axes(ax, 'Principal component reconstruction')
    ax = fig.add_subplot(gs[0, 1]); gray(ax, in_box(dct_recon(Zr_age[ii]))); frame_axes(ax, 'Block cosine reconstruction')
    ax = fig.add_subplot(gs[0, 2]); gray(ax, sl); rec = Us @ (Me[ii] * msd + ml); overlay(ax, paint(rec, brain), 'gray', 1.0, *np.percentile(rec, [1, 99]), upsample=False); frame_axes(ax, 'Surface graph Laplacian reconstruction')
    ax = fig.add_subplot(gs[1, 0]); gray(ax, sl); overlay(ax, np.clip(norm01(gaussian_filter(np.abs(np.tensordot(a_sex[ii] * Zr_sex[ii], pca_imgs, axes=1)), 0.8)), 0, 1)); frame_axes(ax, 'Sex classification')
    ax = fig.add_subplot(gs[1, 1]); gray(ax, sl); overlay(ax, np.clip(dct_attention(a_age[ii], Zr_age[ii]), 0, 1)); frame_axes(ax, 'Age estimation')
    ax = fig.add_subplot(gs[1, 2]); gray(ax, sl); overlay(ax, paint(np.clip(norm01(smooth_surf(np.abs(Us @ (a2[ii] * Me[ii])))), 0, 1), brain), alpha=0.75, upsample=False)
    hip, amy = np.isin(lab, (17, 53)), np.isin(lab, (18, 54)); feet = {'hippocampus': hip, 'amygdala': amy, 'inferior lateral ventricle': horn & ~(hip | amy)}
    o = np.full((S, S), np.nan)
    for k, name in enumerate(MTL): o[feet[name]] = np.clip((SHR[ii, k] - SMIN) / (SMAX - SMIN + 1e-12), 0, 1)
    ax.imshow(o.T, cmap='inferno', origin='lower', vmin=0, vmax=1, alpha=0.8, interpolation='nearest')
    for name in MTL: ax.contour(feet[name].T.astype(float), levels=[0.5], colors=TXT, linewidths=0.35, alpha=0.7, origin='lower')
    frame_axes(ax, 'Cognitive status')
    fig.text(0.5, 0.945, 'Spectral Vision Transformer for Efficient Tokenization with Limited Data', color=TXT, fontsize=14, weight='bold', ha='center')
    fig.text(0.5, ax.get_position().y0 - 0.035, 'Image domain attention projection', color=TXT, fontsize=10, ha='center', va='center')
    fn = f'{FIG}/frame_{j:03d}.png'; fig.savefig(fn, facecolor=BG); plt.close(fig); frames.append(imageio.imread(fn))
imageio.mimsave(f'{FIG}/spectralvit_adni3.gif', frames, duration=130, loop=0)   # milliseconds per frame
print(f'wrote {FIG}/spectralvit_adni3.gif ({len(frames)} frames)')